# EA1 - Actividad 1.4: Arquitecturas para Big Data

## Objetivos
- Comprender el ecosistema Hadoop y su evolucion
- Conocer las arquitecturas Lambda y Kappa
- Comparar soluciones on-premise vs cloud
- Identificar servicios equivalentes en GCP, AWS y Azure

## Conceptos Clave

Este notebook es **mayoritariamente conceptual**. La comprension de las arquitecturas de Big Data es fundamental para disenar sistemas escalables y eficientes.

Las preguntas clave que abordaremos:
- Como almacenamos y procesamos grandes volumenes de datos?
- Como manejamos datos en batch y en tiempo real?
- Cuando elegir on-premise vs cloud?

---
## 1. Ecosistema Hadoop

Hadoop fue el primer framework open-source para procesamiento distribuido de Big Data. Sus componentes principales son:

```
+================================================================+
|                    ECOSISTEMA HADOOP                            |
+================================================================+
|                                                                |
|  +----------+  +----------+  +----------+  +----------+       |
|  |   Hive   |  |   Pig    |  |  Sqoop   |  |  Flume   |       |
|  | (SQL)    |  | (Script) |  | (Import) |  | (Ingest) |       |
|  +----+-----+  +----+-----+  +----+-----+  +----+-----+       |
|       |              |              |              |            |
|  +----v--------------v--------------v--------------v-----+     |
|  |                  MapReduce                            |     |
|  |            (Motor de Procesamiento)                   |     |
|  +------------------------+------------------------------+     |
|                           |                                    |
|  +------------------------v------------------------------+     |
|  |                     YARN                              |     |
|  |          (Gestor de Recursos del Cluster)             |     |
|  +------------------------+------------------------------+     |
|                           |                                    |
|  +------------------------v------------------------------+     |
|  |                     HDFS                              |     |
|  |       (Sistema de Archivos Distribuido)               |     |
|  +-------------------------------------------------------+     |
+================================================================+
```

| Componente | Funcion |
|------------|----------|
| **HDFS** | Sistema de archivos distribuido. Almacena datos en bloques de 128MB replicados en multiples nodos |
| **YARN** | Gestor de recursos. Asigna CPU y memoria a las aplicaciones del cluster |
| **MapReduce** | Modelo de procesamiento: Map (transformar) + Reduce (agregar). Escribe resultados intermedios a disco |
| **Hive** | SQL sobre Hadoop. Traduce consultas SQL a jobs MapReduce |
| **Pig** | Lenguaje de scripting para flujos de datos |
| **Sqoop** | Importar/exportar datos entre HDFS y bases de datos relacionales |
| **Flume** | Ingesta de datos en streaming hacia HDFS |

## 2. Ecosistema Moderno

El ecosistema ha evolucionado significativamente. Hoy en dia, muchas herramientas complementan o reemplazan componentes originales de Hadoop:

```
+-----------------------------------------------------------------+
|                  ECOSISTEMA MODERNO                              |
+-----------------------------------------------------------------+
|                                                                 |
|  Ingesta          Procesamiento       Almacenamiento   Consulta |
|  +--------+       +------------+      +------------+  +------+ |
|  | Kafka  | ----> |   Spark    | ---> |  HDFS /    |  | Hive | |
|  | NiFi   |       |  (Batch +  |      |  S3 /      |  |Presto| |
|  | Flume  |       |  Streaming)|      |  Cloud     |  |Trino | |
|  +--------+       +------------+      | Storage    |  +------+ |
|                                       +------------+           |
|                                                                 |
|  Orquestacion: Airflow, Oozie, Prefect                         |
|  Coordinacion: ZooKeeper                                        |
|  Formato de datos: Parquet, ORC, Avro, Delta Lake               |
+-----------------------------------------------------------------+
```

| Herramienta | Rol | Ventaja sobre Hadoop clasico |
|-------------|-----|------------------------------|
| **Apache Spark** | Procesamiento batch y streaming | 10-100x mas rapido que MapReduce (procesamiento en memoria) |
| **Apache Kafka** | Plataforma de streaming de eventos | Baja latencia, alta capacidad, persistencia de mensajes |
| **Apache Hive** | Data warehouse sobre Hadoop | SQL familiar, optimizado con Tez |
| **Presto/Trino** | Motor de consultas SQL | Consultas interactivas sobre multiples fuentes de datos |
| **Apache Airflow** | Orquestacion de pipelines | DAGs programaticos en Python, monitoreo visual |

---
## 3. Arquitectura Lambda

La **Arquitectura Lambda** fue propuesta por Nathan Marz. Combina procesamiento batch y en tiempo real para ofrecer una vision completa de los datos.

### Diagrama

```
                         +------------------+
                         |   Fuentes de     |
                         |     Datos        |
                         +--------+---------+
                                  |
                     +------------+------------+
                     |                         |
              +------v-------+         +-------v------+
              |  BATCH LAYER |         | SPEED LAYER  |
              |              |         |  (Real-time) |
              | - Almacena   |         | - Procesa    |
              |   todo el    |         |   datos      |
              |   dataset    |         |   recientes  |
              | - Procesa    |         | - Baja       |
              |   en batch   |         |   latencia   |
              | - Alta       |         | - Resultados |
              |   precision  |         |   aprox.     |
              +------+-------+         +-------+------+
                     |                         |
              +------v-------+         +-------v------+
              | Batch Views  |         | Real-time    |
              |              |         | Views        |
              +------+-------+         +-------+------+
                     |                         |
                     +------------+------------+
                                  |
                         +--------v---------+
                         |  SERVING LAYER   |
                         |                  |
                         | Combina batch +  |
                         | real-time views  |
                         | para consultas   |
                         +------------------+
```

### Capas

| Capa | Funcion | Tecnologias tipicas |
|------|---------|--------------------|
| **Batch Layer** | Almacena el dataset completo y genera vistas precalculadas | HDFS, Spark Batch, Hive |
| **Speed Layer** | Procesa datos nuevos en tiempo real para compensar la latencia del batch | Spark Streaming, Storm, Flink |
| **Serving Layer** | Combina resultados de batch y speed para responder consultas | HBase, Cassandra, Druid |

### Ventajas y Desventajas

| Ventajas | Desventajas |
|----------|-------------|
| Tolerante a fallos (el batch corrige errores del speed) | Complejidad: mantener dos pipelines (batch + speed) |
| Resultados precisos (batch) + rapidos (speed) | Duplicacion de logica en dos sistemas diferentes |
| Escalable para grandes volumenes | Mayor costo operacional |
| El batch puede reprocesar todo el historial | Latencia en batch views (horas) |

---
## 4. Arquitectura Kappa

La **Arquitectura Kappa** fue propuesta por Jay Kreps (creador de Kafka). Simplifica Lambda usando **solo streaming** para todo el procesamiento.

### Diagrama

```
     +------------------+
     |   Fuentes de     |
     |     Datos        |
     +--------+---------+
              |
     +--------v---------+
     |  LOG INMUTABLE   |
     |   (ej. Kafka)    |
     | Retiene todo el  |
     | historial de     |
     | eventos          |
     +--------+---------+
              |
     +--------v---------+
     | STREAM PROCESSING|
     |  (ej. Flink,     |
     |   Spark Streaming)|
     |                  |
     | Un solo pipeline |
     | para batch y     |
     | real-time         |
     +--------+---------+
              |
     +--------v---------+
     |  SERVING LAYER   |
     |  Resultados      |
     |  disponibles     |
     |  para consultas  |
     +------------------+
```

### Principio clave
Todo es un **stream de eventos**. Si necesitas reprocesar datos historicos, simplemente re-lees el log desde el inicio.

### Ventajas y Desventajas

| Ventajas | Desventajas |
|----------|-------------|
| Simplicidad: un solo pipeline | Requiere un log inmutable con retencion larga (costo de almacenamiento) |
| Sin duplicacion de logica | Reprocesar grandes volumenes historicos puede ser lento |
| Mas facil de mantener y evolucionar | Menos maduro para algunos escenarios de batch complejo |
| Menor latencia en todos los casos | Depende fuertemente de la tecnologia de streaming |

---
## 5. Comparativa Lambda vs Kappa

| Aspecto | Lambda | Kappa |
|---------|--------|-------|
| **Pipelines** | Dos (batch + speed) | Uno (solo streaming) |
| **Complejidad** | Alta (mantener dos sistemas) | Menor (un solo sistema) |
| **Reprocesamiento** | Batch layer reprocesa todo | Re-leer el log de eventos |
| **Latencia** | Batch: horas / Speed: segundos | Segundos (todo es streaming) |
| **Precision** | Batch corrige errores del speed | Depende de la calidad del streaming |
| **Costo** | Mayor (dos infraestructuras) | Menor infraestructura, mayor almacenamiento de log |
| **Caso ideal** | Necesitas alta precision + tiempo real | Eventos en tiempo real son la fuente de verdad |
| **Ejemplo** | Reportes financieros + dashboard en vivo | Deteccion de fraude, IoT |

---
## 6. On-Premise vs Cloud

| Aspecto | On-Premise | Cloud |
|---------|------------|-------|
| **Costo inicial** | Alto (comprar hardware) | Bajo (pago por uso) |
| **Costo a largo plazo** | Puede ser menor si se usa al maximo | Puede crecer si no se optimiza |
| **Escalabilidad** | Limitada (comprar mas hardware) | Elastica (escalar en minutos) |
| **Mantenimiento** | Equipo propio de infraestructura | El proveedor gestiona el hardware |
| **Seguridad** | Control total | Responsabilidad compartida |
| **Latencia** | Baja (datos locales) | Variable (depende de la region) |
| **Flexibilidad** | Limitada | Alta (servicios managed, serverless) |
| **Time-to-market** | Semanas/meses | Horas/dias |

### Cuando elegir cada opcion?

- **On-Premise:** Regulaciones estrictas de datos (banca, gobierno), carga constante y predecible, ya tienes infraestructura
- **Cloud:** Startups, cargas variables, necesidad de escalar rapidamente, quieres servicios managed
- **Hibrido:** Lo mejor de ambos mundos — datos sensibles on-premise, procesamiento elastico en cloud

---
## 7. Servicios Cloud Equivalentes

Los tres grandes proveedores cloud ofrecen servicios equivalentes para cada componente de una arquitectura Big Data:

| Componente | GCP | AWS | Azure |
|------------|-----|-----|-------|
| **Almacenamiento de objetos** | Cloud Storage | S3 | Blob Storage |
| **Procesamiento distribuido** | Dataproc | EMR | HDInsight |
| **Data Warehouse** | BigQuery | Redshift | Synapse Analytics |
| **Streaming** | Pub/Sub + Dataflow | Kinesis | Event Hubs + Stream Analytics |
| **Orquestacion** | Cloud Composer (Airflow) | Step Functions / MWAA | Data Factory |
| **Base NoSQL** | Bigtable | DynamoDB | Cosmos DB |
| **Catalogo de datos** | Data Catalog | Glue Catalog | Purview |
| **ML Platform** | Vertex AI | SageMaker | Azure ML |
| **Serverless SQL** | BigQuery | Athena | Synapse Serverless |

### Nota sobre costos
Cada proveedor tiene un modelo de precios diferente. Es importante estimar costos antes de elegir, considerando:
- Volumen de datos almacenados
- Cantidad de datos procesados
- Numero de consultas
- Transferencia de datos entre regiones

---
## Ejercicios

Los siguientes ejercicios son **reflexivos**. No requieren codigo, sino analisis y diseno. Escribe tus respuestas en celdas Markdown (cambia el tipo de celda a Markdown) o como comentarios en celdas de codigo.

### EJERCICIO 1: Diseno de arquitectura para e-commerce

**Escenario:** Una empresa de e-commerce tiene:
- Catalogo de 1 millon de productos
- 50.000 ordenes por dia
- Necesidad de recomendaciones de productos en tiempo real
- Reportes diarios de ventas para el equipo de negocio

**TODO:** Disena la arquitectura de datos para este escenario. Responde:
1. Que arquitectura usarias? (Lambda, Kappa, otra)
2. Que componentes/tecnologias elegirais para cada capa?
3. Como manejarias las recomendaciones en tiempo real?
4. Donde almacenarias los datos historicos?
5. Harias on-premise o cloud? Justifica.

Escribe tu respuesta a continuacion (puedes usar esta misma celda o crear nuevas):

### RESPUESTA EJERCICIO 1: Diseno de arquitectura para e-commerce

**1. Arquitectura elegida: Lambda**

Elijo la arquitectura **Lambda** porque el escenario requiere tanto procesamiento batch (reportes diarios de ventas) como procesamiento en tiempo real (recomendaciones de productos). La capa batch garantiza la precision de los reportes, mientras que la capa speed permite recomendaciones con baja latencia.

**2. Componentes por capa:**

- **Ingesta:**
  - **Apache Kafka** como bus de eventos: captura clickstream del usuario, eventos de compra, actualizaciones de inventario en tiempo real.
  - Conectores de bases de datos (Debezium/CDC) para sincronizar cambios del catalogo de productos.

- **Procesamiento batch (Batch Layer):**
  - **Apache Spark** para procesamiento batch diario: calcular metricas de ventas, entrenar modelos de recomendacion, generar reportes agregados.
  - Jobs programados via **Apache Airflow** para orquestar los pipelines batch.

- **Procesamiento real-time (Speed Layer):**
  - **Spark Structured Streaming** o **Apache Flink** para procesar eventos de clickstream en tiempo real.
  - Calcular features en tiempo real del usuario (productos vistos, tiempo en pagina, carrito actual).

- **Almacenamiento:**
  - **Data Lake** en formato Parquet/Delta Lake para datos historicos (almacena todo el historial raw).
  - **Redis** como cache para features de usuario y recomendaciones precalculadas.
  - **PostgreSQL** para el catalogo de productos y datos transaccionales.

- **Serving Layer:**
  - **API REST** que consulta Redis para recomendaciones en tiempo real.
  - **Dashboard** (Metabase/Superset) conectado a las batch views para reportes de negocio.

**3. Recomendaciones en tiempo real:**

El sistema de recomendaciones funciona en dos niveles:
- **Batch:** Un modelo de filtrado colaborativo (ALS en Spark MLlib) se entrena diariamente con todo el historial de compras. Genera recomendaciones precalculadas por usuario y las almacena en Redis.
- **Real-time:** El speed layer actualiza las recomendaciones segun la sesion actual del usuario (productos vistos, busquedas recientes), combinando las recomendaciones batch con el contexto en tiempo real.

**4. Datos historicos:**

Almacenados en un **Data Lake** con formato **Delta Lake** o **Parquet** sobre almacenamiento de objetos (S3/GCS). Estructura en capas: raw (datos crudos tal como llegan), processed (datos limpios y transformados), aggregated (metricas precalculadas). Particionados por fecha para consultas eficientes.

**5. Cloud. Justificacion:**

Recomiendo **cloud** por las siguientes razones:
- Las 50.000 ordenes/dia generan carga variable (picos en eventos como Black Friday, CyberDay).
- La escalabilidad elastica del cloud permite manejar estos picos sin sobredimensionar hardware permanente.
- Servicios managed (EMR/Dataproc, Managed Kafka) reducen la carga operacional.
- Time-to-market mas rapido: no hay que esperar semanas para aprovisionar servidores.
- Proveedor sugerido: **GCP** con Dataproc (Spark), Pub/Sub (streaming), BigQuery (reportes), Cloud Storage (data lake).

> **Nota docente:** Este ejercicio no tiene una unica respuesta correcta. Lo importante es que el estudiante justifique sus decisiones. Puntos a evaluar: (1) coherencia entre la arquitectura elegida y los requisitos del escenario; (2) correcta identificacion de componentes por capa; (3) justificacion de on-premise vs cloud. Una respuesta que elija Kappa tambien es valida si argumenta que los reportes diarios pueden generarse a partir del stream (usando ventanas de tiempo). Fomentar la discusion en clase sobre las alternativas.

### EJERCICIO 2: Arquitectura Lambda para IoT industrial

**Escenario:** Una planta industrial tiene:
- 500 sensores que envian datos cada segundo (temperatura, presion, vibracion)
- Necesidad de detectar anomalias en tiempo real para prevenir fallas
- Reportes mensuales de mantenimiento predictivo
- Almacenamiento de datos historicos por 5 anos (regulacion)

**TODO:** Identifica los componentes de una arquitectura Lambda para este sistema:
1. Que tecnologia usarias para la ingesta de datos de los sensores?
2. Que iria en el Batch Layer? Que procesamiento haria?
3. Que iria en el Speed Layer? Como detectarias anomalias en tiempo real?
4. Que iria en el Serving Layer? Como se consultarian los datos?
5. Que formato de almacenamiento usarias para los datos historicos y por que?

### RESPUESTA EJERCICIO 2: Arquitectura Lambda para IoT industrial

**1. Ingesta de datos de sensores:**

- **Apache Kafka** como plataforma de ingesta principal. Cada sensor publica mensajes a un topic de Kafka (ej: `sensor-temperatura`, `sensor-presion`, `sensor-vibracion` o un topic unificado `sensor-data`).
- Protocolo de comunicacion: **MQTT** en los sensores (ligero, ideal para IoT) con un broker MQTT (Mosquitto) que reenvia a Kafka mediante un conector Kafka Connect.
- Con 500 sensores enviando datos cada segundo, se generan ~43 millones de registros diarios, lo cual Kafka maneja sin problemas.

**2. Batch Layer:**

- **Almacenamiento:** Todos los datos raw se almacenan en **HDFS** o **Cloud Storage** en formato **Parquet** (columnar, comprimido, ideal para series temporales). Particionado por `fecha/sensor_id` para consultas eficientes.
- **Procesamiento:** **Apache Spark** ejecuta jobs batch programados con **Airflow**:
  - Calculo de estadisticas agregadas por sensor (promedios diarios, mensuales, desviaciones estandar).
  - Entrenamiento/re-entrenamiento de modelos de mantenimiento predictivo (regression, random forest sobre features historicas).
  - Generacion de reportes mensuales de mantenimiento.
- **Output:** Batch views con metricas precalculadas y predicciones de falla para cada equipo, almacenadas en **Hive** o **BigQuery** para consultas SQL.

**3. Speed Layer:**

- **Procesamiento:** **Spark Structured Streaming** o **Apache Flink** consume datos directamente de Kafka en micro-batches o procesamiento evento-a-evento.
- **Deteccion de anomalias:** Reglas en tiempo real combinadas con modelo de ML:
  - Reglas estaticas: alertar si temperatura > umbral critico, presion fuera de rango, vibracion excesiva.
  - Modelo estadistico: detectar desviaciones de mas de 3 sigmas respecto a la media movil del sensor (usando ventanas deslizantes).
  - Modelo ML (opcional): modelo ligero pre-entrenado (entrenado en batch layer) aplicado en streaming para deteccion mas sofisticada.
- **Alertas:** Las anomalias detectadas se publican en un topic de Kafka dedicado (`alertas`) y se envian via **PagerDuty** o notificaciones push al equipo de mantenimiento.

**4. Serving Layer:**

- **Base de datos:** **Apache Druid** o **InfluxDB** (optimizadas para series temporales) para datos recientes y consultas en tiempo real. **HBase** o **Cassandra** para datos historicos con acceso rapido por clave (sensor_id + timestamp).
- **API/Dashboard:** **Grafana** conectado a la base de series temporales para visualizacion en tiempo real de los sensores. Dashboard personalizado con alertas activas, estado de cada sensor y predicciones de mantenimiento.

**5. Formato de almacenamiento historico:**

**Apache Parquet** por las siguientes razones:
- **Columnar:** Las consultas analiticas sobre series temporales tipicamente acceden a pocas columnas (ej: solo temperatura), y Parquet permite leer solo las columnas necesarias.
- **Compresion:** Parquet con Snappy/ZSTD comprime muy bien datos numericos repetitivos de sensores, reduciendo costos de almacenamiento para 5 anos de datos.
- **Compatible:** Spark, Hive, Presto/Trino y la mayoria de herramientas del ecosistema leen Parquet nativamente.
- **Particionado:** Permite particionar por fecha, lo cual facilita la politica de retencion (eliminar particiones con mas de 5 anos) y acelera consultas por rango de tiempo.

> **Nota docente:** En este ejercicio se pide explicitar una arquitectura Lambda. Evaluar que el estudiante identifique correctamente las tres capas y asigne tecnologias apropiadas a cada una. Puntos extra si mencionan: (1) el protocolo MQTT como estandar para IoT; (2) la importancia del formato Parquet para series temporales; (3) el calculo del volumen de datos (500 sensores * 1 msg/s * 86400 s/dia); (4) la politica de retencion de 5 anos y como implementarla. Un error comun es confundir la capa de ingesta con la speed layer: Kafka es ingesta, el procesamiento streaming es la speed layer.

---
## Resumen

En esta actividad aprendimos:

1. **Ecosistema Hadoop:** HDFS (almacenamiento), YARN (recursos), MapReduce (procesamiento) + herramientas como Hive, Pig, Sqoop
2. **Ecosistema moderno:** Spark, Kafka, Airflow y formatos como Parquet y Delta Lake
3. **Arquitectura Lambda:** Tres capas (Batch + Speed + Serving) para combinar precision y velocidad
4. **Arquitectura Kappa:** Un solo pipeline de streaming, simplicidad a costa de cierta flexibilidad
5. **On-Premise vs Cloud:** Trade-offs entre control, costo, escalabilidad y time-to-market
6. **Servicios cloud:** GCP, AWS y Azure ofrecen componentes equivalentes para cada necesidad

---
## Desafio Extra (Opcional)

**Arquitectura para deteccion de fraude bancario en tiempo real**

**Escenario:** Un banco procesa 10 millones de transacciones diarias con tarjeta de credito. Necesita:
- Detectar transacciones fraudulentas en menos de 500ms
- Modelos de ML que se actualizan diariamente con nuevos patrones de fraude
- Dashboard en tiempo real para el equipo de seguridad
- Almacenamiento de todas las transacciones por 7 anos (regulacion bancaria)
- Cumplimiento de regulaciones de privacidad de datos

**TODO:**
1. Elegiras Lambda o Kappa? Justifica tu decision
2. Disena la arquitectura completa con componentes especificos
3. Como integras el modelo de ML en el pipeline de deteccion?
4. Como manejas el reentrenamiento diario del modelo?
5. Que consideraciones de seguridad y privacidad tendrias?

### RESPUESTA DESAFIO: Arquitectura para deteccion de fraude bancario

**1. Arquitectura elegida: Lambda. Justificacion:**

Elijo **Lambda** por las siguientes razones:
- Se requiere deteccion en menos de 500ms (speed layer con modelo pre-entrenado).
- Se requiere actualizacion diaria del modelo con nuevos patrones (batch layer para reentrenamiento).
- Los reportes historicos y auditorias regulatorias necesitan precision absoluta (batch layer garantiza esto).
- La capa batch puede recalcular toda la historia si se descubre un nuevo patron de fraude, corrigiendo falsos negativos historicos.

Una arquitectura Kappa seria viable, pero la complejidad del reentrenamiento de modelos de ML y los requisitos regulatorios de precision hacen que Lambda sea mas apropiada.

**2. Componentes de la arquitectura:**

- **Ingesta de transacciones:**
  - **Apache Kafka** con multiples particiones para manejar 10M transacciones/dia (~115 transacciones/segundo promedio, con picos de hasta 1000/s).
  - Topics: `transacciones-raw`, `transacciones-scored`, `alertas-fraude`.
  - Garantia de entrega: `acks=all` para no perder ninguna transaccion.

- **Procesamiento real-time (Speed Layer):**
  - **Apache Flink** (preferido sobre Spark Streaming por su latencia sub-segundo y procesamiento evento a evento).
  - Pipeline: recibe transaccion -> enriquece con features del cliente (desde Redis) -> aplica modelo ML -> emite score de fraude.
  - Si score > umbral: bloquear transaccion y generar alerta.
  - Latencia objetivo: < 200ms para dejar margen al umbral de 500ms.

- **Procesamiento batch (Batch Layer):**
  - **Apache Spark** para:
    - Reentrenamiento diario del modelo de deteccion de fraude con datos etiquetados (fraudes confirmados + transacciones legitimas).
    - Calculo de features agregadas por cliente (patrones de gasto mensual, comercios frecuentes, horarios tipicos).
    - Reportes de auditoria y compliance.
  - Orquestado por **Apache Airflow**.

- **Modelo de ML (serving):**
  - Modelo entrenado: **XGBoost** o **LightGBM** (rapidos en inferencia, buenos para datos tabulares).
  - Servido via **MLflow** + contenedor Docker para versionamiento y despliegue.
  - Almacenado en un **Model Registry** para trazabilidad de versiones.

- **Almacenamiento:**
  - **Data Lake** en formato Delta Lake sobre almacenamiento de objetos para los 7 anos de historial.
  - **Redis** para cache de features de cliente en tiempo real.
  - **PostgreSQL** para metadatos, configuraciones y resultados de auditoria.

- **Dashboard:**
  - **Grafana** para metricas operacionales en tiempo real (transacciones/segundo, tasa de fraude, latencia del sistema).
  - **Apache Superset** o herramienta BI interna para el equipo de seguridad (investigacion de casos, drill-down en alertas).

**3. Integracion del modelo de ML en el pipeline:**

El modelo se integra en la speed layer como un componente del pipeline de Flink:
1. La transaccion llega a Flink desde Kafka.
2. Flink enriquece la transaccion con features pre-calculadas del cliente (desde Redis): promedio de gasto, ubicacion habitual, frecuencia de transacciones.
3. Se calculan features en tiempo real: distancia geografica respecto a la ultima transaccion, tiempo desde la ultima transaccion, monto relativo al promedio del cliente.
4. El vector de features se pasa al modelo pre-cargado en memoria (serializado con joblib/pickle).
5. El modelo retorna un score de probabilidad de fraude (0 a 1).
6. Si score > 0.8: bloquear y alertar. Si score entre 0.5-0.8: alertar para revision manual. Si score < 0.5: aprobar.

**4. Reentrenamiento diario del modelo:**

Pipeline batch diario orquestado por Airflow:
1. **Extraer** transacciones del ultimo dia con etiquetas de fraude confirmado (feedback del equipo de seguridad).
2. **Feature engineering:** calcular features historicas actualizadas para cada cliente.
3. **Entrenar** nuevo modelo con datos historicos + nuevos datos etiquetados. Usar validacion cruzada temporal (entrenar con datos antiguos, validar con datos recientes).
4. **Evaluar** el nuevo modelo vs el modelo en produccion (comparar precision, recall, F1, AUC). Solo promover si mejora las metricas.
5. **Desplegar** el nuevo modelo al Model Registry y actualizarlo en los nodos de Flink mediante hot-swap (sin downtime).
6. **Monitorear** drift del modelo: alertar si la distribucion de scores cambia significativamente.

**5. Consideraciones de seguridad y privacidad:**

- **Encriptacion:** Datos encriptados en transito (TLS/mTLS entre todos los componentes) y en reposo (AES-256 en almacenamiento).
- **Anonimizacion:** Los datos de entrenamiento del modelo deben ser pseudonimizados (reemplazar datos personales por IDs hasheados). No almacenar nombres ni numeros de tarjeta completos en el data lake.
- **Control de acceso:** RBAC (Role-Based Access Control) estricto. El equipo de seguridad ve alertas, el equipo de datos ve datos anonimizados, solo sistemas automatizados acceden a datos sensibles.
- **Auditoria:** Log inmutable de todas las decisiones del modelo (que transacciones se bloquearon y por que). Esto es obligatorio para regulaciones como PCI-DSS.
- **Retencion:** Politica de retencion de 7 anos para transacciones (regulacion bancaria). Implementar borrado automatico de datos personales asociados cuando sea requerido (derecho al olvido, donde aplique).
- **Residencia de datos:** Si se usa cloud, asegurar que los datos residan en la region correcta segun la regulacion local (ej: datos de clientes chilenos en servidores en Chile o Latinoamerica).
- **On-premise vs Hibrido:** Para un banco, lo ideal es una arquitectura **hibrida**: datos sensibles y modelo de scoring on-premise (cumplimiento regulatorio), con capacidad de burst al cloud para reentrenamiento y procesamiento batch pesado usando datos anonimizados.

> **Nota docente:** Este desafio es el mas complejo del notebook y evalua la capacidad de integrar multiples conceptos. No se espera que los estudiantes lleguen a este nivel de detalle, pero si que cubran al menos: (1) justificacion de la arquitectura; (2) componentes principales de cada capa; (3) consideraciones basicas de seguridad. Puntos clave para la discusion en clase: la tension entre latencia y precision, el concepto de model registry y versionamiento de modelos, la diferencia entre encriptacion en transito vs en reposo, y por que un banco tipicamente usa arquitectura hibrida. Este ejercicio puede servir como base para un trabajo grupal mas extenso.